# Lab Instructions

You have been hired by James Cameron to create profiles of two characters for a reboot of the Titanic Movie: one that is most likely to survive the sinking and one that is least likely to survive.  Mr. Cameron wants this reboot to be as historically accurate as possible, so your profile of each character should be backed up with data and visualizations.

Each character profile should include information on their:
* Age, fare
* Sex
* Passenger class
* Travel companions (including both parents/children and siblings/spouse)
* Port of departure (indicated by the Embarked feature in the dataset)

For quantitative features like `Age` and `Fare`, you will need to use the `.loc` method we learned in class (or something similar) to place individuals in categories.  How you choose to do this is up to you, but make sure you explain your reasoning.

You should include at least one visualization for each element of the character profile (age, sex, passenger class, etc.) as evidence.

After you have developed your two character profiles, use your Pandas data wrangling skills to identify at least one real passenger in the dataset that fits each profile.  Print out the names of these individuals.  Look them up in [Encyclopeida Titanica](https://www.encyclopedia-titanica.org/) (or a similar resource).  

Tell Mr. Cameron at least one thing about the real passengers who fit your two character profiles that you learned from an external resource.  You need one interesting fact about a person who fits the profile of "most likely to survive" and one interesting fact about a person who fits the profile of "least likely to surivive".  



In [212]:
import pandas as pd

df = pd.read_csv('titanic_passengers.csv')



In [213]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [214]:
df['Fare'].describe()

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64

In [215]:
pd.crosstab(df['Sex'], df['Survived'])

Survived,0,1
Sex,,
female,81,233
male,468,109


In [216]:
pd.crosstab(df['Survived'], df['Sex'], normalize=True)*100


Sex,female,male
Survived,,
0,9.090909,52.525253
1,26.150393,12.233446


In [217]:
pd.crosstab(df['Pclass'], df['Survived'], normalize='index')*100

Survived,0,1
Pclass,,
1,37.037037,62.962963
2,52.717391,47.282609
3,75.763747,24.236253


So far,
Least Likely: Male, Pclass = 3
Most Likely: Female, Pclass = 1

In [218]:
df['Cabin'].isna().value_counts()

Cabin
True     687
False    204
Name: count, dtype: int64

In [219]:
df['Cabin_Status'] = 'Cabin Present'
df.loc[df['Cabin'].isna(), 'Cabin_Status'] = 'No Cabin Info'
df['Cabin_Status'].value_counts()

Cabin_Status
No Cabin Info    687
Cabin Present    204
Name: count, dtype: int64

In [220]:
pd.crosstab(df['Cabin_Status'], df['Survived'], normalize=True)*100

Survived,0,1
Cabin_Status,,
Cabin Present,7.631874,15.263749
No Cabin Info,53.984287,23.120090


In [221]:
pd.crosstab(df['Cabin_Status'], df['Survived'], normalize='index')*100

Survived,0,1
Cabin_Status,,
Cabin Present,33.333333,66.666667
No Cabin Info,70.014556,29.985444


--------------------------
Having No Cabin Assignment made you MUCH less likely to survive.

In [222]:
df['Embarked'].value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [223]:
pd.crosstab(df['Embarked'], df['Survived'], normalize='index')*100

Survived,0,1
Embarked,,
C,44.642857,55.357143
Q,61.038961,38.961039
S,66.304348,33.695652


In [224]:
pd.crosstab(df['Embarked'], df['Survived'], normalize=True)*100

Survived,0,1
Embarked,,
C,8.436445,10.461192
Q,5.286839,3.374578
S,48.031496,24.409449


So now we have:

Least Likely: Male, Pclass = 3, No cabin status, Embark = S

Most Likely Female, Pclass = 1, Cabin Assigned, Embark = C

In [225]:
df['Fare_Bin'] = pd.cut(df['Fare'], bins=[0, 10, 20, 30, 1000], labels=['Low Fare', 'Mid Fare', 'High Fare', 'Very High Fare'])
df['Fare_Bin'].value_counts()

Fare_Bin
Low Fare          321
Very High Fare    234
Mid Fare          179
High Fare         142
Name: count, dtype: int64

In [226]:
pd.crosstab(df['Fare_Bin'], df['Survived'], normalize='index')*100

Survived,0,1
Fare_Bin,,
Low Fare,79.439252,20.560748
Mid Fare,57.541899,42.458101
High Fare,55.633803,44.366197
Very High Fare,41.880342,58.119658


In [227]:
df['SibSp'].value_counts()

SibSp
0    608
1    209
2     28
4     18
3     16
8      7
5      5
Name: count, dtype: int64

In [228]:
df['SibSp_Status'] = 'Alone'
df.loc[df['SibSp'] > 0, 'SibSp_Status'] = 'With Siblings/Spouses'
df['SibSp_Status'].value_counts()

SibSp_Status
Alone                    608
With Siblings/Spouses    283
Name: count, dtype: int64

In [229]:
pd.crosstab(df['SibSp_Status'], df['Survived'], normalize='index')*100

Survived,0,1
SibSp_Status,,
Alone,65.460526,34.539474
With Siblings/Spouses,53.356890,46.643110


So now we have:

Least Likely: Male, Pclass = 3, No cabin status, Embark = S, Low-Fare, No SibSp

Most Likely Female, Pclass = 1, Cabin Assigned, Embark = C, Very High-Fare, With SibSp

In [230]:
df['ParCh_Status'] = 'No Family'
df.loc[df['Parch'] > 0, 'ParCh_Status'] = 'With Parents/Children'
df['ParCh_Status'].value_counts()

ParCh_Status
No Family                678
With Parents/Children    213
Name: count, dtype: int64

In [231]:
pd.crosstab(df['ParCh_Status'], df['Survived'], normalize='index')*100

Survived,0,1
ParCh_Status,,
No Family,65.634218,34.365782
With Parents/Children,48.826291,51.173709


In [232]:
df['Age_Bin'] = pd.cut(df['Age'], bins=[0, 18, 35, 55, 130], labels=['Child', 'Adult', 'Middle Aged', 'Senior'])
df['Age_Bin'].value_counts()

Age_Bin
Adult          358
Middle Aged    177
Child          139
Senior          40
Name: count, dtype: int64

In [233]:
pd.crosstab(df['Age_Bin'], df['Survived'], normalize='index')*100

Survived,0,1
Age_Bin,,
Child,49.640288,50.359712
Adult,61.731844,38.268156
Middle Aged,59.887006,40.112994
Senior,70.000000,30.000000


Finally we have:

Least Likely: Male, Pclass = 3, No cabin status, Embark = S, Low-Fare, No SibSp, No Parch, Senior (age >= 55)

Most Likely Female, Pclass = 1, Cabin Assigned, Embark = C, Very High-Fare, With SibSp, With Parch, Child (age <= 12 )

In [242]:
# Define strict masks for both profiles
least_mask = (
    (df['Sex'] == 'male') &
    (df['Pclass'] == 3) &
    (df['Cabin'].isna() | (df['Cabin'] == '')) &
    (df['Embarked'] == 'S') &
    (df['Fare'] <= 10) &
    (df['SibSp'] == 0) &
    (df['Parch'] == 0) &
    (df['Age'] >= 55)
)

most_mask = (
    (df['Sex'] == 'female') &
    (df['Pclass'] == 1) &
    (~df['Cabin'].isna() & (df['Cabin'] != '')) &
    (df['Fare'] >= 20) &
    (df['SibSp'] != 0) &
    (df['Parch'] != 0) &
    (df['Age'] <= 18)
)

# Assign Survival_Profile
df_out = df.copy()
df_out['Survival_Profile'] = 'Other'
df_out.loc[least_mask, 'Survival_Profile'] = 'Least_Likely'
df_out.loc[most_mask, 'Survival_Profile'] = 'Most_Likely'

# Extract matching rows
least_df = df_out[df_out['Survival_Profile'] == 'Least_Likely']
most_df = df_out[df_out['Survival_Profile'] == 'Most_Likely']

print(df_out['Survival_Profile'].value_counts())
display(most_df.head())
display(least_df.head())

Survival_Profile
Other           884
Least_Likely      4
Most_Likely       3
Name: count, dtype: int64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Cabin_Status,Fare_Bin,SibSp_Status,ParCh_Status,Age_Bin,Survival_Profile
297,298,0,1,"Allison, Miss. Helen Loraine",female,2.0,1,2,113781,151.550,C22 C26,S,Cabin Present,Very High Fare,With Siblings/Spouses,With Parents/Children,Child,Most_Likely
311,312,1,1,"Ryerson, Miss. Emily Borie",female,18.0,2,2,PC 17608,262.375,B57 B59 B63 B66,C,Cabin Present,Very High Fare,With Siblings/Spouses,With Parents/Children,Child,Most_Likely
435,436,1,1,"Carter, Miss. Lucile Polk",female,14.0,1,2,113760,120.000,B96 B98,S,Cabin Present,Very High Fare,With Siblings/Spouses,With Parents/Children,Child,Most_Likely


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Cabin_Status,Fare_Bin,SibSp_Status,ParCh_Status,Age_Bin,Survival_Profile
94,95,0,3,"Coxon, Mr. Daniel",male,59.0,0,0,364500,7.2500,NaN,S,No Cabin Info,Low Fare,Alone,No Family,Senior,Least_Likely
152,153,0,3,"Meo, Mr. Alfonzo",male,55.5,0,0,A.5. 11206,8.0500,NaN,S,No Cabin Info,Low Fare,Alone,No Family,Senior,Least_Likely
326,327,0,3,"Nysveen, Mr. Johan Hansen",male,61.0,0,0,345364,6.2375,NaN,S,No Cabin Info,Low Fare,Alone,No Family,Senior,Least_Likely
851,852,0,3,"Svensson, Mr. Johan",male,74.0,0,0,347060,7.7750,NaN,S,No Cabin Info,Low Fare,Alone,No Family,Senior,Least_Likely


So we have a list of 4 Least-likely to survive and 3 Most_likely to survive.

I will choose:

Miss Lucile Polk Carter for Most Likely to survive. Lucile Polk was a prominent figure born into wealth, with her paternal ancestor being the 11th US President, President Polk. She was also a prominent member of the Baltimore Fashion and Cosmopolitan circles. Her survival on the sinkin was shared with by all her children as she paddled the survival dingy away from the sinking ship. Miraculously, she was able to locate her husband and they all survived. Although they were given the miracle of surviving somehting so brutal, she later experience severe abuse by her husband, including injuries from a "horse-whip". They later divorced.

Mr. Johan Svensson for Least likely to survive. Johan Svensson was a common farmer from Sweden who moved to North Dakota with his son after his wife passed away. He was the oldest person aboard the Titanic, but he and his son were both lost in the sinking. His fare ticket costed ~7 pounds.

Both stories were sourced from Encyclopedia Titanica website.

